# 01: Embeddings Basics

Learn how to generate text embeddings using the Gemini API, understand their structure, and visualize them in 2D.

## Prerequisites

- Python 3.10+
- `openai` installed (`pip install openai`)
- `scikit-learn` and `matplotlib` for visualization
- A OpenAI API key from [OpenAI Platform](https://platform.openai.com/apikey)

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
if not API_KEY:
    print("WARNING: OPENAI_API_KEY not found. Create a .env file with your key.")
else:
    print(f"API Key loaded: {API_KEY[:8]}...")

In [ ]:
import openai
import numpy as np

API_KEY = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=API_KEY)

EMBEDDING_MODEL = "models/text-embedding-3-small"
print(f"Using embedding model: {EMBEDDING_MODEL}")

## 2. Generate Your First Embedding

An embedding is a list of numbers (floats) that represents the meaning of a text.

In [ ]:
result = genai.embed_content(
    model=EMBEDDING_MODEL,
    content="What is machine learning?"
)

embedding = result["embedding"]
print(f"Embedding type: {type(embedding)}")
print(f"Embedding length: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"\nFull embedding (first 20): {np.array(embedding[:20]).round(4)}")

### Understanding Dimensions

The `text-embedding-3-small` model produces **768-dimensional** vectors. Each dimension captures some aspect of meaning, though we can't easily interpret what each dimension means.

In [ ]:
embedding_array = np.array(embedding)

print(f"Vector shape: {embedding_array.shape}")
print(f"Vector magnitude (norm): {np.linalg.norm(embedding_array):.4f}")
print(f"Min value: {embedding_array.min():.4f}")
print(f"Max value: {embedding_array.max():.4f}")
print(f"Mean value: {embedding_array.mean():.4f}")
print(f"Std deviation: {embedding_array.std():.4f}")

## 3. Batch Embeddings

Generate embeddings for multiple texts at once (more efficient than one-by-one).

In [ ]:
texts = [
    "What is machine learning?",
    "How does deep learning work?",
    "Explain neural networks",
    "What is the weather today?",
    "How do I cook pasta?",
]

results = genai.embed_content(
    model=EMBEDDING_MODEL,
    content=texts
)

embeddings = np.array(results["embedding"])
print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {embeddings.shape[1]} dimensions")

## 4. Comparing Similar vs Different Sentences

Similar sentences should have embeddings that are close together in vector space.

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


pairs = [
    ("I love programming", "I enjoy coding"),
    ("I love programming", "The cat sat on the mat"),
    ("The stock market crashed", "Financial markets declined sharply"),
    ("The stock market crashed", "I like to eat pizza"),
]

for text_a, text_b in pairs:
    result_a = genai.embed_content(model=EMBEDDING_MODEL, content=text_a)
    result_b = genai.embed_content(model=EMBEDDING_MODEL, content=text_b)
    
    emb_a = np.array(result_a["embedding"])
    emb_b = np.array(result_b["embedding"])
    
    sim = cosine_similarity(emb_a, emb_b)
    print(f"Similarity: {sim:.4f}")
    print(f"  A: {text_a}")
    print(f"  B: {text_b}")
    print()

### Expected Results

| Pair | Expected Similarity | Why |
|------|-------------------|-----|
| love programming / enjoy coding | High (~0.8+) | Similar meaning |
| love programming / cat on mat | Low (~0.3) | Unrelated topics |
| stock market crashed / markets declined | High (~0.9+) | Very similar meaning |
| stock market crashed / eat pizza | Very low (~0.1) | Completely unrelated |

## 5. Visualizing Embeddings in 2D

We can't visualize 768 dimensions, but we can project embeddings to 2D using PCA or t-SNE.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Sample texts across different topics
sample_texts = [
    "What is machine learning?",
    "How does deep learning work?",
    "Explain neural networks",
    "What are transformers in AI?",
    "What is the weather today?",
    "Will it rain tomorrow?",
    "How do I cook pasta?",
    "Best recipes for dinner",
    "How to invest in stocks?",
    "Stock market analysis",
    "Learn Python programming",
    "Python tutorial for beginners",
]

labels = [
    "AI", "AI", "AI", "AI",
    "Weather", "Weather",
    "Food", "Food",
    "Finance", "Finance",
    "Programming", "Programming",
]

# Generate embeddings
results = genai.embed_content(model=EMBEDDING_MODEL, content=sample_texts)
embeddings = np.array(results["embedding"])
print(f"Generated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}")

In [ ]:
# PCA visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Plot
colors = {
    "AI": "#FF6B6B",
    "Weather": "#4ECDC4",
    "Food": "#45B7D1",
    "Finance": "#96CEB4",
    "Programming": "#FFEAA7",
}

fig, ax = plt.subplots(figsize=(10, 8))

for label in set(labels):
    mask = [l == label for l in labels]
    ax.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=colors[label],
        label=label,
        s=100,
        alpha=0.7,
    )

# Add text labels
for i, text in enumerate(sample_texts):
    ax.annotate(
        text[:25] + "..." if len(text) > 25 else text,
        (embeddings_2d[i, 0], embeddings_2d[i, 1]),
        fontsize=8,
        alpha=0.7,
    )

ax.set_title("Embeddings Visualized with PCA")
ax.legend()
plt.tight_layout()
plt.show()

### Exercise: t-SNE Visualization

t-SNE often produces better cluster separations than PCA. Try it below:

In [ ]:
# Exercise: Use t-SNE to visualize the same embeddings
# Hint: from sklearn.manifold import TSNE
# tsne = TSNE(n_components=2, random_state=42, perplexity=5)
# embeddings_tsne = tsne.fit_transform(embeddings)

# TODO: Create a scatter plot similar to the PCA plot above
# Compare which visualization shows better clustering

## 6. Embeddings as Search

Find the most relevant text for a query using cosine similarity.

In [ ]:
# Document collection
documents = [
    "Python is a popular programming language for data science.",
    "Machine learning models need large datasets for training.",
    "The weather forecast predicts rain this weekend.",
    "Neural networks are inspired by the human brain.",
    "FastAPI is a modern Python web framework.",
    "Climate change affects global weather patterns.",
    "Deep learning has revolutionized computer vision.",
    "JavaScript is essential for web development.",
]

# Generate document embeddings
doc_results = genai.embed_content(model=EMBEDDING_MODEL, content=documents)
doc_embeddings = np.array(doc_results["embedding"])

In [ ]:
def search(query, documents, doc_embeddings, top_k=3):
    """Search documents by semantic similarity."""
    # Embed the query
    query_result = genai.embed_content(model=EMBEDDING_MODEL, content=query)
    query_embedding = np.array(query_result["embedding"])
    
    # Compute similarities
    similarities = np.array([
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ])
    
    # Sort by similarity
    top_indices = similarities.argsort()[::-1][:top_k]
    
    return [(documents[i], similarities[i]) for i in top_indices]


# Test searches
queries = [
    "artificial intelligence",
    "web development",
    "climate and environment",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = search(query, documents, doc_embeddings)
    for doc, score in results:
        print(f"  [{score:.4f}] {doc}")

## 7. Task Type Parameter

Gemini embeddings support a `task_type` parameter that optimizes embeddings for specific use cases.

In [ ]:
# Different task types
task_types = ["RETRIEVAL_QUERY", "RETRIEVAL_DOCUMENT", "SEMANTIC_SIMILARITY"]

text = "How to learn machine learning"

for task in task_types:
    result = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type=task,
    )
    emb = np.array(result["embedding"])
    print(f"Task: {task:30s} | Norm: {np.linalg.norm(emb):.4f} | First 3 values: {emb[:3].round(4)}")

## 8. Reducing Dimensions

You can request lower-dimensional embeddings for faster computation.

In [ ]:
dimensions = [256, 512, 768]

for dim in dimensions:
    result = genai.embed_content(
        model=EMBEDDING_MODEL,
        content="What is machine learning?",
        output_dimensionality=dim,
    )
    emb = np.array(result["embedding"])
    print(f"Requested: {dim} | Actual: {len(emb)} dimensions")

## Exercises

1. **Similarity ranking**: Create a list of 10 sentences about different topics. For a given query, rank all sentences by similarity. Do the results make sense?

2. **Dimension comparison**: Generate embeddings at 256, 512, and 768 dimensions for the same set of texts. Compute pairwise similarities at each dimension. Do the rankings change?

3. **Task type impact**: For a search use case, compare embeddings generated with `RETRIEVAL_QUERY` vs `RETRIEVAL_DOCUMENT` vs no task type. Which gives better search results?

4. **Visualization experiment**: Add 5 more sentences of your choice to the visualization. Do they cluster with the expected group?

## Summary

- Text embeddings are dense vectors that capture semantic meaning
- Similar texts produce similar embeddings (close in vector space)
- Gemini `text-embedding-3-small` produces 768-dimensional vectors
- Batch processing is more efficient than individual calls
- PCA/t-SNE can visualize high-dimensional embeddings in 2D
- Cosine similarity measures how similar two embeddings are

→ Next: [Similarity Search](02-similarity-search.ipynb)